# 🌿 Doctor Green — ResNet50 Training on Google Colab

## What this notebook does:
1. Connects to Kaggle and downloads the Plant Disease Dataset (~2.7 GB)
2. Trains a ResNet50 model using transfer learning
3. Evaluates accuracy on validation set
4. Saves `doctor_green_model.pt` and `class_names.json`
5. Downloads both files to your computer

## After training:
- Copy `doctor_green_model.pt` → `doctor-green/backend/model/`
- Copy `class_names.json` → `doctor-green/backend/data/`
- Run `python app.py` — real AI predictions will work!

---
**⚡ IMPORTANT: Before running, go to:**
`Runtime → Change runtime type → Hardware accelerator → T4 GPU`
---

## Step 1 — Upload your Kaggle API Key

1. Go to https://www.kaggle.com → Your Profile → Settings → API → Create New Token
2. This downloads `kaggle.json` to your computer
3. Run the cell below and upload that file when prompted

In [ ]:
from google.colab import files
import os

print('📁 Upload your kaggle.json file now...')
uploaded = files.upload()

# Move to correct location
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ kaggle.json configured!')

## Step 2 — Install Dependencies & Download Dataset

In [ ]:
# Install kaggle
!pip install kaggle -q

# Download dataset (~2.7 GB — takes 3-5 minutes)
print('⬇️  Downloading New Plant Diseases Dataset...')
print('   This takes 3-5 minutes on Colab...')
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/dataset
print('✅ Download complete!')

In [ ]:
# Unzip dataset
import zipfile, os

zip_path = '/content/dataset/new-plant-diseases-dataset.zip'
print('📦 Extracting dataset...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/dataset')
os.remove(zip_path)
print('✅ Extracted!')

# Find train/valid directories
train_dir = None
valid_dir = None
for root, dirs, files in os.walk('/content/dataset'):
    if 'train' in dirs and 'valid' in dirs:
        train_dir = os.path.join(root, 'train')
        valid_dir = os.path.join(root, 'valid')
        break

print(f'📂 Train dir: {train_dir}')
print(f'📂 Valid dir: {valid_dir}')

# Count classes
classes = sorted(os.listdir(train_dir))
print(f'🌿 Classes found: {len(classes)}')
print('Classes:', classes)

## Step 3 — Check GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

## Step 4 — Setup DataLoaders

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import json, os, time

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH     = 32
EPOCHS    = 10       # Increase to 15-20 for ~98% accuracy
LR        = 0.001
WORKERS   = 2

# Data transforms
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
val_ds   = datasets.ImageFolder(valid_dir, transform=val_tf)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=WORKERS, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=True)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'✅ Train: {len(train_ds):,} images | Val: {len(val_ds):,} images | Classes: {NUM_CLASSES}')

## Step 5 — Build ResNet50 Model

In [ ]:
# Load pretrained ResNet50
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze all layers except layer4 and fc — speeds up training
for name, param in model.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

# Replace final layer for our 38 classes
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ Model ready — Trainable params: {trainable:,} / {total:,}')

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
print('✅ Optimizer and scheduler configured')

## Step 6 — Train the Model

⏱️ **Expected time on T4 GPU: ~45 minutes for 10 epochs**

Expected accuracy: **95–98%** on validation set

In [ ]:
from tqdm.notebook import tqdm

best_val_acc = 0.0
history = []
MODEL_SAVE_PATH = '/content/doctor_green_model.pt'

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='  Train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
        correct  += out.argmax(1).eq(labels).sum().item()
        total    += labels.size(0)
    return loss_sum / len(loader), 100.0 * correct / total

def validate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='  Val  ', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labels)
            loss_sum += loss.item()
            correct  += out.argmax(1).eq(labels).sum().item()
            total    += labels.size(0)
    return loss_sum / len(loader), 100.0 * correct / total

print('🚀 Training started...')
print('=' * 65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_dl, criterion, optimizer)
    vl_loss, vl_acc = validate(model, val_dl, criterion)
    scheduler.step()
    elapsed = time.time() - t0

    history.append({'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc,
                    'vl_loss': vl_loss, 'vl_acc': vl_acc})

    saved = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        saved = ' ← SAVED ✅'

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train: {tr_acc:.2f}% (loss {tr_loss:.4f}) | '
          f'Val: {vl_acc:.2f}% (loss {vl_loss:.4f}) | '
          f'{elapsed:.0f}s{saved}')

print('=' * 65)
print(f'🏆 Best Validation Accuracy: {best_val_acc:.2f}%')
print(f'💾 Model saved to: {MODEL_SAVE_PATH}')

## Step 7 — Save Class Names

In [ ]:
CLASS_NAMES_PATH = '/content/class_names.json'
with open(CLASS_NAMES_PATH, 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

print(f'✅ Saved {len(CLASS_NAMES)} class names to {CLASS_NAMES_PATH}')
print('Classes:')
for i, c in enumerate(CLASS_NAMES):
    print(f'  {i:02d}. {c}')

## Step 8 — Verify Model Works

In [ ]:
# Reload model from disk and test on one validation image
import random
from PIL import Image

# Reload
test_model = models.resnet50(weights=None)
test_model.fc = nn.Linear(test_model.fc.in_features, NUM_CLASSES)
test_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location='cpu'))
test_model.eval()
print('✅ Model reloaded from disk successfully')

# Pick a random validation image
random_class = random.choice(CLASS_NAMES)
class_dir = os.path.join(valid_dir, random_class)
random_img_file = random.choice(os.listdir(class_dir))
img_path = os.path.join(class_dir, random_img_file)

img = Image.open(img_path).convert('RGB')
tensor = val_tf(img).unsqueeze(0)

with torch.no_grad():
    probs = torch.softmax(test_model(tensor), dim=1)[0]

top3_vals, top3_idx = probs.topk(3)
print(f'\n📸 Test image: {random_class}/{random_img_file}')
print(f'\n🔍 Top 3 Predictions:')
for i in range(3):
    predicted_class = CLASS_NAMES[top3_idx[i].item()]
    confidence = top3_vals[i].item() * 100
    match = '✅ CORRECT' if predicted_class == random_class else ''
    print(f'  {i+1}. {predicted_class}: {confidence:.2f}% {match}')

print(f'\n  Actual class: {random_class}')

## Step 9 — Download the 2 Files to Your Computer

**These are the ONLY 2 files you need to copy into your Doctor Green project.**

In [ ]:
from google.colab import files
import os

model_size = os.path.getsize(MODEL_SAVE_PATH) / 1024 / 1024
print(f'📦 doctor_green_model.pt — {model_size:.1f} MB')
print(f'📄 class_names.json     — {os.path.getsize(CLASS_NAMES_PATH) / 1024:.1f} KB')
print()
print('⬇️  Downloading doctor_green_model.pt ...')
files.download(MODEL_SAVE_PATH)

print('⬇️  Downloading class_names.json ...')
files.download(CLASS_NAMES_PATH)

print()
print('=' * 55)
print('✅ DONE! Now do this on your computer:')
print()
print('  1. Move doctor_green_model.pt  →  doctor-green/backend/model/')
print('  2. Move class_names.json       →  doctor-green/backend/data/')
print('  3. cd doctor-green/backend')
print('  4. python app.py')
print('  5. cd ../frontend && npm start')
print()
print('  🌿 Your app will now give REAL AI predictions!')
print('=' * 55)

## ✅ Summary

| File | Destination in your project |
|------|-----------------------------|
| `doctor_green_model.pt` | `doctor-green/backend/model/` |
| `class_names.json` | `doctor-green/backend/data/` |

That's it! The Flask backend (`app.py`) automatically detects and loads these files.